In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import keras
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

In [2]:
dataset_train = pd.read_csv('D:/Books/project/0005-store-sales-time-series-forecasting/train.csv')
dataset_holidays_events = pd.read_csv('D:/Books/project/0005-store-sales-time-series-forecasting/holidays_events.csv')
dataset_oil = pd.read_csv('D:/Books/project/0005-store-sales-time-series-forecasting/oil.csv')
dataset_stores = pd.read_csv('D:/Books/project/0005-store-sales-time-series-forecasting/stores.csv')
dataset_transactions = pd.read_csv('D:/Books/project/0005-store-sales-time-series-forecasting/transactions.csv')

In [3]:

Dataset = pd.merge(dataset_train,dataset_holidays_events,on='date',how='left')
Dataset = pd.merge(Dataset , dataset_oil , on='date',how='left')
Dataset = pd.merge(Dataset , dataset_transactions , on=['date', 'store_nbr'],how='left')
Dataset = pd.merge(Dataset , dataset_stores , on='store_nbr',how='left')

In [4]:
le_family = LabelEncoder()
le_type_x = LabelEncoder()
le_locale = LabelEncoder()
le_locale_name = LabelEncoder()
le_description = LabelEncoder()
le_city = LabelEncoder()
le_state = LabelEncoder()
le_type_y = LabelEncoder()

Dataset['date'] = pd.to_datetime(Dataset['date'])
Dataset['Year']=Dataset['date'].dt.year
Dataset['Month']=Dataset['date'].dt.month
Dataset['Day']=Dataset['date'].dt.day

Dataset['family']= le_family.fit_transform(Dataset['family'])

Dataset['type_x']= le_type_x.fit_transform(Dataset['type_x'])

Dataset['locale']= le_locale.fit_transform(Dataset['locale'])

Dataset['locale_name']= le_locale_name.fit_transform(Dataset['locale_name'])

Dataset['description']= le_description.fit_transform(Dataset['description'])

Dataset['transferred']=Dataset['transferred'].fillna(False)
Dataset['transferred']=Dataset['transferred'].replace({False:0,True:1})

Dataset['dcoilwtico']=Dataset['dcoilwtico'].ffill().bfill()

Dataset['transactions']=Dataset['transactions'].fillna(0)

Dataset['city']= le_city.fit_transform(Dataset['city'])

Dataset['state']= le_state.fit_transform(Dataset['state'])

Dataset['type_y']= le_type_y.fit_transform(Dataset['type_y'])

Dataset=Dataset.drop('date',axis=1)
Dataset=Dataset.astype(float)

In [5]:
Dataset

,id,store_nbr,family,sales,onpromotion,type_x,locale,locale_name,description,transferred,dcoilwtico,transactions,city,state,type_y,cluster,Year,Month,Day
0,0.0,1.0,0.0,0.000,0.0,3.0,1.0,4.0,50.0,0.0,93.14,0.0,18.0,12.0,3.0,13.0,2013.0,1.0,1.0
1,1.0,1.0,1.0,0.000,0.0,3.0,1.0,4.0,50.0,0.0,93.14,0.0,18.0,12.0,3.0,13.0,2013.0,1.0,1.0
2,2.0,1.0,2.0,0.000,0.0,3.0,1.0,4.0,50.0,0.0,93.14,0.0,18.0,12.0,3.0,13.0,2013.0,1.0,1.0
3,3.0,1.0,3.0,0.000,0.0,3.0,1.0,4.0,50.0,0.0,93.14,0.0,18.0,12.0,3.0,13.0,2013.0,1.0,1.0
4,4.0,1.0,4.0,0.000,0.0,3.0,1.0,4.0,50.0,0.0,93.14,0.0,18.0,12.0,3.0,13.0,2013.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3054343,3000883.0,9.0,28.0,438.133,0.0,3.0,0.0,19.0,28.0,0.0,47.57,2155.0,18.0,12.0,1.0,6.0,2017.0,8.0,15.0
3054344,3000884.0,9.0,29.0,154.553,1.0,3.0,0.0,19.0,28.0,0.0,47.57,2155.0,18.0,12.0,1.0,6.0,2017.0,8.0,15.0
3054345,3000885.0,9.0,30.0,2419.729,148.0,3.0,0.0,19.0,28.0,0.0,47.57,2155.0,18.0,12.0,1.0,6.0,2017.0,8.0,15.0
3054346,3000886.0,9.0,31.0,121.000,8.0,3.0,0.0,19.0,28.0,0.0,47.57,2155.0,18.0,12.0,1.0,6.0,2017.0,8.0,15.0


In [6]:
x = Dataset.drop('sales',axis=1)
y = Dataset['sales']

x =StandardScaler().fit_transform(x)

x_train , x_test , y_train , y_test = train_test_split(x , y ,random_state=42 , test_size=0.2)

In [7]:
modell = XGBRegressor().fit(x_train,y_train)
print(modell.score(x_test,y_test))
print(r2_score(modell.predict(x_test),y_test))

0.9327482353853436
0.9271023206646734


In [8]:
model = keras.Sequential([
    keras.layers.Dense(256,input_shape=(18,),activation='relu'),
    keras.layers.Dense(128,input_shape=(18,),activation='relu'),
    keras.layers.Dense(64,activation='relu'),
    keras.layers.Dense(32,activation='relu'),
    keras.layers.Dense(1)
])
model.compile(loss='mae' , optimizer='adam' , metrics=['mae'])

history = model.fit(
    x_train, 
    y_train,
    epochs=20, 
    batch_size=256,  
    validation_split=0.2,
    verbose=1
)

c:\Users\mahdi\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/20
7636/7636 ━━━━━━━━━━━━━━━━━━━━ 36s 4ms/step - loss: 229.1315 - mae: 229.1315 - val_loss: 163.7102 - val_mae: 163.7102
Epoch 2/20
7636/7636 ━━━━━━━━━━━━━━━━━━━━ 33s 4ms/step - loss: 146.9079 - mae: 146.9079 - val_loss: 137.8316 - val_mae: 137.8316
Epoch 3/20
7636/7636 ━━━━━━━━━━━━━━━━━━━━ 35s 5ms/step - loss: 124.4386 - mae: 124.4386 - val_loss: 118.0771 - val_mae: 118.0771
Epoch 4/20
7636/7636 ━━━━━━━━━━━━━━━━━━━━ 32s 4ms/step - loss: 107.2392 - mae: 107.2392 - val_loss: 98.8790 - val_mae: 98.8790
Epoch 5/20
7636/7636 ━━━━━━━━━━━━━━━━━━━━ 36s 5ms/step - loss: 93.7157 - mae: 93.7157 - val_loss: 90.4900 - val_mae: 90.4900
Epoch 6/20
7636/7636 ━━━━━━━━━━━━━━━━━━━━ 34s 4ms/step - loss: 84.0650 - mae: 84.0650 - val_loss: 80.7607 - val_mae: 80.7607
Epoch 7/20
7636/7636 ━━━━━━━━━━━━━━━━━━━━ 34s 4ms/step - loss: 75.1344 - mae: 75.1344 - val_loss: 74.8218 - val_mae: 74.8218
Epoch 8/20
7636/7636 ━━━━━━━━━━━━━━━━━━━━ 35s 5ms/step - loss: 71.2285 - mae: 71.2285 - val_loss: 71.4684 - val

In [28]:
print(r2_score(model.predict(x_test),y_test))
model.save('D:/Books/project/0005-store-sales-time-series-forecasting/keras.keras')

19090/19090 ━━━━━━━━━━━━━━━━━━━━ 27s 1ms/step
0.9414123165009411


In [29]:
dataset_test = pd.read_csv('D:/Books/project/0005-store-sales-time-series-forecasting/test.csv')

Dataset_test = pd.merge(dataset_test,dataset_holidays_events,on='date',how='left')
Dataset_test = pd.merge(Dataset_test , dataset_oil , on='date',how='left')
Dataset_test = pd.merge(Dataset_test , dataset_transactions , on=['date', 'store_nbr'],how='left')
Dataset_test = pd.merge(Dataset_test , dataset_stores , on='store_nbr',how='left')

In [30]:
Dataset_test['date'] = pd.to_datetime(Dataset_test['date'])
Dataset_test['Year']=Dataset_test['date'].dt.year
Dataset_test['Month']=Dataset_test['date'].dt.month
Dataset_test['Day']=Dataset_test['date'].dt.day

Dataset_test['family']= le_family.transform(Dataset_test['family'])

Dataset_test['type_x']= le_type_x.transform(Dataset_test['type_x'])

Dataset_test['locale']= le_locale.transform(Dataset_test['locale'])

Dataset_test['locale_name']= le_locale_name.transform(Dataset_test['locale_name'])

Dataset_test['description']= le_description.transform(Dataset_test['description'])

Dataset_test['transferred']=Dataset_test['transferred'].fillna(False)
Dataset_test['transferred']=Dataset_test['transferred'].replace({False:0,True:1})

Dataset_test['dcoilwtico']=Dataset_test['dcoilwtico'].ffill().bfill()

Dataset_test['transactions']=Dataset_test['transactions'].fillna(0)

Dataset_test['city']= le_city.transform(Dataset_test['city'])

Dataset_test['state']= le_state.transform(Dataset_test['state'])

Dataset_test['type_y']= le_type_y.transform(Dataset_test['type_y'])

Dataset_test=Dataset_test.drop('date',axis=1)
Dataset_test=Dataset_test.astype(float)

Dataset_test['id']=Dataset_test['id'].astype(int)

In [31]:
Dataset_test

,id,store_nbr,family,onpromotion,type_x,locale,locale_name,description,transferred,dcoilwtico,transactions,city,state,type_y,cluster,Year,Month,Day
0,3000888,1.0,0.0,0.0,6.0,3.0,24.0,101.0,0.0,46.80,0.0,18.0,12.0,3.0,13.0,2017.0,8.0,16.0
1,3000889,1.0,1.0,0.0,6.0,3.0,24.0,101.0,0.0,46.80,0.0,18.0,12.0,3.0,13.0,2017.0,8.0,16.0
2,3000890,1.0,2.0,2.0,6.0,3.0,24.0,101.0,0.0,46.80,0.0,18.0,12.0,3.0,13.0,2017.0,8.0,16.0
3,3000891,1.0,3.0,20.0,6.0,3.0,24.0,101.0,0.0,46.80,0.0,18.0,12.0,3.0,13.0,2017.0,8.0,16.0
4,3000892,1.0,4.0,0.0,6.0,3.0,24.0,101.0,0.0,46.80,0.0,18.0,12.0,3.0,13.0,2017.0,8.0,16.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28507,3029395,9.0,28.0,1.0,6.0,3.0,24.0,101.0,0.0,47.26,0.0,18.0,12.0,1.0,6.0,2017.0,8.0,31.0
28508,3029396,9.0,29.0,0.0,6.0,3.0,24.0,101.0,0.0,47.26,0.0,18.0,12.0,1.0,6.0,2017.0,8.0,31.0
28509,3029397,9.0,30.0,1.0,6.0,3.0,24.0,101.0,0.0,47.26,0.0,18.0,12.0,1.0,6.0,2017.0,8.0,31.0
28510,3029398,9.0,31.0,9.0,6.0,3.0,24.0,101.0,0.0,47.26,0.0,18.0,12.0,1.0,6.0,2017.0,8.0,31.0


In [ ]:
sales = model.predict(Dataset_test)


891/891 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step


array([[1.7609492e+07],
       [1.7608676e+07],
       [1.7609008e+07],
       ...,
       [1.7753848e+07],
       [1.7757240e+07],
       [1.7752052e+07]], shape=(28512, 1), dtype=float32)

In [42]:
fff=pd.DataFrame({'id':Dataset_test['id'],'sales':model.predict(Dataset_test).ravel()})
fff.to_csv('D:/Books/project/0005-store-sales-time-series-forecasting/fff.csv',index=False)

891/891 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
